# PJBL Data Analytics — Inspect & Prepare (base Steam)

## Base: `fronkongames/games.csv`

Este notebook aplica à base real do projeto (tema Steam) o mesmo fluxo usado na
demonstração de aula com o Titanic: **Carregar → Conhecer → Inspecionar →
Identificar problemas → Decidir → Transformar → Validar**.

> **Pergunta central:** recebemos os dados — mas podemos confiar neles?

Diferente da demonstração didática, aqui **não introduzimos problema nenhum
artificialmente**. Todo problema documentado abaixo foi encontrado no dado
real, com evidência (contagem, amostra, coluna e linha) antes de qualquer
tratamento.

## Preparação do ambiente

Usamos apenas Pandas — a base já tem volume e sujeira suficientes para a
análise sem depender de bibliotecas de visualização.

In [1]:
import csv
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

CAMINHO_ORIGINAL = "../Dataset/fronkongames/games.csv"
CAMINHO_LIMPO = "../Dataset/fronkongames/games_clean.csv"

print("Ambiente preparado.")

Ambiente preparado.


# 1. Carregar

Ingestão simples com `pandas.read_csv`, exatamente como qualquer pessoa faria
na primeira tentativa — sem nenhum parâmetro especial. É esse carregamento
"ingênuo" que vai revelar o primeiro problema real da base, na seção de
inspeção.

In [2]:
df_bruto = pd.read_csv(CAMINHO_ORIGINAL)

print(f"Shape (leitura ingênua): {df_bruto.shape[0]} linhas × {df_bruto.shape[1]} colunas")
display(df_bruto.head(3))

Shape (leitura ingênua): 125855 linhas × 39 colunas


,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,Full audio languages,Reviews,Header image,Website,Support url,Support email,Windows,Mac,Linux,Metacritic score,Metacritic url,User score,Positive,Negative,Score rank,Achievements,Recommendations,Notes,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,[],[],NaN,https://shared.akamai.steamstatic.com/store_it...,NaN,NaN,NaN,True,False,False,0,NaN,0,0,0,NaN,0,0,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",['English'],[],NaN,https://shared.akamai.steamstatic.com/store_it...,http://mangagamer.org/supipara,http://mangagamer.com,support@mangagamer.com,True,False,False,0,NaN,0,252,3,NaN,0,231,NaN,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",[],NaN,https://shared.akamai.steamstatic.com/store_it...,https://www.facebook.com/8FloorGames/,https://www.facebook.com/8FloorGames,support@8floor.net,True,True,False,0,NaN,0,21,3,NaN,0,0,NaN,0,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN


# 2. Conhecer

Dicionário resumido das colunas mais relevantes para o projeto (dataset tem
39 colunas nominais no cabeçalho; o significado completo de cada uma está na
página do dataset no Kaggle — `fronkongames/steam-games-dataset`).

In [3]:
dicionario = pd.DataFrame({
    "coluna": ["AppID", "Name", "Release date", "Estimated owners", "Price",
               "Metacritic score", "Genres", "Developers", "Windows/Mac/Linux",
               "Positive", "Negative"],
    "significado": [
        "Identificador único do jogo na Steam",
        "Nome do jogo",
        "Data de lançamento",
        "Faixa estimada de compradores (ex.: '0 - 20000')",
        "Preço em USD",
        "Nota agregada do Metacritic (0 quando não há nota)",
        "Gêneros do jogo (lista como texto)",
        "Estúdio(s) responsável(is)",
        "Plataformas suportadas (booleano)",
        "Quantidade de reviews positivas",
        "Quantidade de reviews negativas",
    ],
})
display(dicionario.style.hide(axis="index"))

coluna,significado
AppID,Identificador único do jogo na Steam
Name,Nome do jogo
Release date,Data de lançamento
Estimated owners,Faixa estimada de compradores (ex.: '0 - 20000')
Price,Preço em USD
Metacritic score,Nota agregada do Metacritic (0 quando não há nota)
Genres,Gêneros do jogo (lista como texto)
Developers,Estúdio(s) responsável(is)
Windows/Mac/Linux,Plataformas suportadas (booleano)
Positive,Quantidade de reviews positivas


# 3. Inspecionar

Começamos pela visão estrutural padrão: shape, tipos inferidos e uma amostra
de linhas.

In [4]:
print("VISÃO GERAL — LEITURA INGÊNUA")
print("─" * 50)
print(f"Linhas:  {df_bruto.shape[0]}")
print(f"Colunas: {df_bruto.shape[1]}")
print()
df_bruto.info()

VISÃO GERAL — LEITURA INGÊNUA
──────────────────────────────────────────────────
Linhas:  125855
Colunas: 39

<class 'pandas.DataFrame'>
Index: 125855 entries, 2539430 to 4247310
Data columns (total 39 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   AppID                       125854 non-null  str    
 1   Name                        125855 non-null  str    
 2   Release date                125855 non-null  str    
 3   Estimated owners            125855 non-null  int64  
 4   Peak CCU                    125855 non-null  int64  
 5   Required age                125855 non-null  float64
 6   Price                       125855 non-null  int64  
 7   DiscountDLC count           125855 non-null  int64  
 8   About the game              117393 non-null  str    
 9   Supported languages         125855 non-null  str    
 10  Full audio languages        125855 non-null  str    
 11  Reviews                     1

## Problema 1 — Cabeçalho malformado desalinha TODAS as colunas a partir de "About the game"

Ao inspecionar `df_bruto`, algo não fecha: a coluna `Release date` traz
valores como `"0 - 20000"` (formato de faixa, não de data), e a coluna `AppID`
tem 1 valor nulo — inesperado para um identificador. Isso indica desalinhamento
de colunas, não sujeira nos dados em si. Investigamos a causa comparando o
número de campos do cabeçalho com o número de campos de uma linha de dado real,
usando o módulo `csv` (que respeita aspas e vírgulas internas corretamente).

In [5]:
# Evidência do desalinhamento: valores de "Release date" que deveriam ser datas
print("Amostra de 'Release date' na leitura ingênua (deveria ser data):")
print(df_bruto["Release date"].sample(8, random_state=2).tolist())
print()
print(f"AppID nulo (não deveria existir, é o identificador): {df_bruto['AppID'].isnull().sum()}")

Amostra de 'Release date' na leitura ingênua (deveria ser data):
['200000 - 500000', '0 - 0', '0 - 20000', '0 - 20000', '0 - 20000', '0 - 20000', '0 - 20000', '20000 - 50000']

AppID nulo (não deveria existir, é o identificador): 1


In [6]:
# Comparação direta cabeçalho vs. linha de dado, via csv.reader (aspas tratadas corretamente)
with open(CAMINHO_ORIGINAL, newline="", encoding="utf-8") as f:
    leitor = csv.reader(f)
    cabecalho = next(leitor)
    primeira_linha = next(leitor)

print(f"Campos no cabeçalho: {len(cabecalho)}")
print(f"Campos na 1ª linha de dado: {len(primeira_linha)}")
print()
print("Trecho do cabeçalho ao redor do ponto suspeito:")
print(cabecalho[6:10])

Campos no cabeçalho: 39
Campos na 1ª linha de dado: 40

Trecho do cabeçalho ao redor do ponto suspeito:
['Price', 'DiscountDLC count', 'About the game', 'Supported languages']


**Causa confirmada:** o cabeçalho tem **39 nomes de coluna**, mas cada
linha de dado tem **40 campos**. O nome `"DiscountDLC count"` é, na verdade,
a fusão de dois nomes de coluna que perderam o separador (`"Discount"` +
`"DLC count"`) — sobrou um campo de dado sem nome correspondente no cabeçalho.
A partir daí, o Pandas empurra cada valor uma posição para a coluna errada:
o texto de `About the game` aparece sob `Supported languages`, o preço aparece
deslocado, os booleanos de plataforma (`Windows`/`Mac`/`Linux`) caem em colunas
erradas, e assim por diante — para as **125.855 linhas**, não só para casos
isolados.

**Por que é um problema:** qualquer análise feita em cima da leitura ingênua
(preço médio, gêneros mais comuns, plataformas suportadas etc.) estaria lendo
a coluna errada. É o problema mais grave da base porque contamina silenciosamente
quase todas as colunas ao mesmo tempo.

**Solução proposta:** reconstruir a lista de nomes de coluna inserindo o nome
que faltava (`"Discount"`) na posição correta e reler o CSV com esse cabeçalho
corrigido, em vez de usar a primeira linha do arquivo como cabeçalho.

In [7]:
# Reconstrói o cabeçalho: insere o nome que faltava logo após "DiscountDLC count"
cabecalho_corrigido = cabecalho[:8] + ["Discount"] + cabecalho[8:]
print(f"Cabeçalho original:  {len(cabecalho)} nomes")
print(f"Cabeçalho corrigido: {len(cabecalho_corrigido)} nomes (bate com os {len(primeira_linha)} campos de dado)")

df_original = pd.read_csv(CAMINHO_ORIGINAL, skiprows=1, header=None, names=cabecalho_corrigido)

print(f"\nShape (leitura corrigida): {df_original.shape[0]} linhas × {df_original.shape[1]} colunas")
print(f"AppID nulo agora: {df_original['AppID'].isnull().sum()}")
display(df_original[["AppID", "Name", "Release date", "Price", "About the game", "Windows", "Mac", "Linux"]].head(3))

Cabeçalho original:  39 nomes
Cabeçalho corrigido: 40 nomes (bate com os 40 campos de dado)



Shape (leitura corrigida): 125855 linhas × 40 colunas
AppID nulo agora: 0


,AppID,Name,Release date,Price,About the game,Windows,Mac,Linux
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0.00,NaN,True,False,False
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",5.24,"Springtime, April: when the cherry trees come ...",True,False,False
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",4.99,"Immerse yourself in the most beloved, mystical...",True,True,False


Com o cabeçalho corrigido, `Release date` volta a conter datas de verdade,
`About the game` traz o texto de descrição, os booleanos de plataforma fazem
sentido, e não há mais `AppID` nulo. **Todo o restante da inspeção usa
`df_original` (já com colunas alinhadas), não `df_bruto`.**

## Problemas 2 a 6 — inspeção do dataset já realinhado

In [8]:
ausencias = pd.DataFrame({
    "quantidade ausente": df_original.isna().sum(),
    "percentual ausente (%)": df_original.isna().mean().mul(100),
}).sort_values("percentual ausente (%)", ascending=False)

print("VALORES AUSENTES (top 15 colunas)")
display(ausencias.head(15).style.format({"percentual ausente (%)": "{:.1f}%"}))

VALORES AUSENTES (top 15 colunas)


,quantidade ausente,percentual ausente (%)
Movies,125855,100.0%
Score rank,125815,100.0%
Metacritic url,121597,96.6%
Reviews,113674,90.3%
Notes,102545,81.5%
Website,75335,59.9%
Support url,70560,56.1%
Tags,42502,33.8%
Support email,22630,18.0%
Categories,8966,7.1%


### Problema 2 — coluna `Movies` 100% vazia

A coluna `Movies` (trailers/vídeos do jogo) não tem um único valor preenchido
em nenhuma das 125.855 linhas.

**Por que é um problema:** uma coluna inteiramente nula não carrega informação
nenhuma — só ocupa espaço e pode confundir quem for explorar a base achando
que existe dado de vídeo disponível.

**Solução proposta:** remover a coluna.

In [9]:
print(f"Movies — nulos: {df_original['Movies'].isnull().sum()} / {len(df_original)} linhas")
print(f"Movies — valores não nulos: {df_original['Movies'].notna().sum()}")

Movies — nulos: 125855 / 125855 linhas
Movies — valores não nulos: 0


### Problema 3 — `Estimated owners` é uma faixa em texto, não um número

A coluna vem como string no formato `"0 - 20000"`, `"20000 - 50000"` etc.

**Por que é um problema:** não dá pra somar, ordenar numericamente ou comparar
essa coluna sem antes convertê-la — qualquer `.describe()` ou correlação com
preço/reviews ignora essa coluna por ser texto.

**Solução proposta:** extrair limite inferior e superior como colunas
numéricas (`owners_min`, `owners_max`) e usar a média dos dois como
`owners_avg`, uma estimativa pontual utilizável em análises numéricas.

In [10]:
print("Formato observado (10 categorias mais comuns):")
display(df_original["Estimated owners"].value_counts().head(10).to_frame("ocorrências"))

Formato observado (10 categorias mais comuns):


,ocorrências
Estimated owners,
0 - 20000,75705
0 - 0,24579
20000 - 50000,11397
50000 - 100000,5355
100000 - 200000,3454
200000 - 500000,2856
500000 - 1000000,1155
1000000 - 2000000,729
2000000 - 5000000,405


### Problema 4 — `Metacritic score` usa 0 como sentinela de "sem nota", não como nota real

**Por que é um problema:** 0 é um valor numérico válido dentro da escala
(0–100), então médias e distribuições calculadas direto sobre a coluna tratam
"não avaliado" como se fosse "nota mínima", distorcendo qualquer estatística.

**Solução proposta:** converter `0` para `NaN` nessa coluna (jogo sem nota
real de Metacritic é a leitura correta; nota real igual a zero é praticamente
inexistente na prática do site).

In [11]:
total = len(df_original)
zerados = int((df_original["Metacritic score"] == 0).sum())
print(f"Metacritic score == 0: {zerados} de {total} linhas ({zerados/total:.1%})")
print(f"Distribuição das notas != 0:")
display(df_original.loc[df_original["Metacritic score"] != 0, "Metacritic score"].describe())

Metacritic score == 0: 121597 de 125855 linhas (96.6%)
Distribuição das notas != 0:


count    4258.000000
mean       73.895726
std        10.262581
min         6.000000
25%        68.000000
50%        76.000000
75%        81.000000
max        97.000000
Name: Metacritic score, dtype: float64

### Problema 5 — valores ausentes em colunas categóricas usadas para análise

`Genres`, `Developers`, `Categories`, `Publishers` e `Tags` têm ausências
relevantes (de ~8 mil a ~42 mil linhas, a depender da coluna).

**Por que é um problema:** ao agrupar por gênero/desenvolvedora (ex.: "jogos
por gênero", "preço médio por desenvolvedora") essas linhas seriam
silenciosamente excluídas do agrupamento sem nenhum aviso.

**Solução proposta:** preencher com o marcador explícito `"Not specified"` —
mantém a linha disponível para contagens gerais (ex.: total de jogos, preço
médio) sem fingir um gênero/desenvolvedora que não existe no dado original.

In [12]:
for coluna in ["Genres", "Developers", "Categories", "Publishers", "Tags"]:
    n = df_original[coluna].isnull().sum()
    print(f"{coluna:12s}: {n:6d} ausentes ({n/total:.1%})")

Genres      :   8423 ausentes (6.7%)
Developers  :   8449 ausentes (6.7%)
Categories  :   8966 ausentes (7.1%)
Publishers  :   8922 ausentes (7.1%)
Tags        :  42502 ausentes (33.8%)


### Problema 6 — nomes de jogo duplicados (mesmo título, `AppID` diferente)

**Por que é um problema:** pode inflar contagens de "quantidade de jogos" se
alguém assumir erroneamente que `Name` é único. Diferente de uma duplicata
exata de linha, aqui não dá pra simplesmente apagar — pode ser demo + jogo
completo, remaster, ou reedição regional, todos com `AppID` (identificador
real) diferente e legítimo.

**Solução proposta:** não remover automaticamente (removeria jogos válidos);
documentar a contagem e manter `AppID` como chave única da base, não `Name`.

In [13]:
nomes_normalizados = df_original["Name"].astype(str).str.strip().str.lower()
duplicados_nome = int(nomes_normalizados.duplicated().sum())
duplicados_linha_exata = int(df_original.duplicated().sum())
duplicados_appid = int(df_original["AppID"].duplicated().sum())

print(f"Duplicatas de linha exata (todas as colunas iguais): {duplicados_linha_exata}")
print(f"Duplicatas de AppID (identificador):                 {duplicados_appid}")
print(f"Duplicatas de Name (título repetido, AppID distinto): {duplicados_nome}")
print()
print("Exemplo de títulos repetidos com AppID diferente:")
exemplo = df_original[nomes_normalizados.isin(
    nomes_normalizados[nomes_normalizados.duplicated()].unique()[:3]
)][["AppID", "Name", "Release date"]].sort_values("Name")
display(exemplo.head(9))

Duplicatas de linha exata (todas as colunas iguais): 0
Duplicatas de AppID (identificador):                 0
Duplicatas de Name (título repetido, AppID distinto): 1593

Exemplo de títulos repetidos com AppID diferente:


,AppID,Name,Release date
1202,3437200,LUNA,"Sep 24, 2025"
3934,1450100,LUNA,"Nov 20, 2020"
997,1122690,Last Stop,"Jul 22, 2021"
2411,2327980,Last Stop,"Mar 13, 2023"
42286,605770,Luna,"Oct 17, 2017"
24315,666810,Luna,"Aug 15, 2017"
115274,905320,Shadow of the Tomb Raider: Definitive Edition,"Sep 14, 2018"
101947,849179,Shadow of the Tomb Raider: Definitive Edition,"Sep 14, 2018"
101347,849164,Shadow of the Tomb Raider: Definitive Edition,"Sep 14, 2018"


# 5. Decidir

Antes de transformar, registramos cada decisão e sua justificativa.

In [14]:
decisoes = pd.DataFrame({
    "problema": [
        "Cabeçalho malformado (colunas desalinhadas)",
        "Coluna Movies 100% vazia",
        "Estimated owners como faixa em texto",
        "Metacritic score usando 0 como sentinela",
        "Ausências em colunas categóricas",
        "Duplicatas de linha exata",
        "Nomes de jogo repetidos (AppID distinto)",
    ],
    "decisão": [
        "Reler o CSV com cabeçalho corrigido (já aplicado na inspeção)",
        "Remover a coluna",
        "Extrair owners_min, owners_max e owners_avg numéricos",
        "Converter 0 → NaN",
        "Preencher com 'Not specified'",
        "Remover (se houver) via drop_duplicates()",
        "Manter; AppID continua sendo a chave única, não Name",
    ],
    "justificativa": [
        "Sem isso quase toda coluna carrega o valor da coluna vizinha",
        "Zero informação, zero utilidade analítica",
        "Necessário pra qualquer estatística numérica de popularidade",
        "0 como nota real distorce médias e distribuições",
        "Evita excluir jogos de agrupamentos por engano",
        "Linha 100% igual a outra é redundância pura",
        "Remover destruiria demos/remasters/reedições legítimas",
    ],
})
display(decisoes.style.hide(axis="index"))

problema,decisão,justificativa
Cabeçalho malformado (colunas desalinhadas),Reler o CSV com cabeçalho corrigido (já aplicado na inspeção),Sem isso quase toda coluna carrega o valor da coluna vizinha
Coluna Movies 100% vazia,Remover a coluna,"Zero informação, zero utilidade analítica"
Estimated owners como faixa em texto,"Extrair owners_min, owners_max e owners_avg numéricos",Necessário pra qualquer estatística numérica de popularidade
Metacritic score usando 0 como sentinela,Converter 0 → NaN,0 como nota real distorce médias e distribuições
Ausências em colunas categóricas,Preencher com 'Not specified',Evita excluir jogos de agrupamentos por engano
Duplicatas de linha exata,Remover (se houver) via drop_duplicates(),Linha 100% igual a outra é redundância pura
Nomes de jogo repetidos (AppID distinto),"Manter; AppID continua sendo a chave única, não Name",Remover destruiria demos/remasters/reedições legítimas


# 6. Transformar

Cada tratamento parte de `df_original` (já com colunas alinhadas) e grava o
resultado em `df_limpo`, mostrando o estado antes/depois.

In [15]:
df_limpo = df_original.copy(deep=True)
print(f"Shape inicial de df_limpo: {df_limpo.shape}")

Shape inicial de df_limpo: (125855, 40)


In [16]:
# TRATAMENTO 1 — remover a coluna Movies (100% vazia)
colunas_antes = df_limpo.shape[1]
df_limpo = df_limpo.drop(columns=["Movies"])
print("REMOÇÃO DA COLUNA MOVIES")
print("─" * 50)
print(f"Colunas antes:  {colunas_antes}")
print(f"Colunas depois: {df_limpo.shape[1]}")
print("✓ Coluna 100% vazia removida.")

REMOÇÃO DA COLUNA MOVIES
──────────────────────────────────────────────────
Colunas antes:  40
Colunas depois: 39
✓ Coluna 100% vazia removida.


In [17]:
# TRATAMENTO 2 — Estimated owners: faixa em texto -> colunas numéricas
faixa = df_limpo["Estimated owners"].str.split(" - ", expand=True)
df_limpo["owners_min"] = pd.to_numeric(faixa[0], errors="coerce")
df_limpo["owners_max"] = pd.to_numeric(faixa[1], errors="coerce")
df_limpo["owners_avg"] = (df_limpo["owners_min"] + df_limpo["owners_max"]) / 2

print("EXTRAÇÃO NUMÉRICA DE ESTIMATED OWNERS")
print("─" * 50)
print(f"owners_min/max/avg ausentes: {df_limpo[['owners_min','owners_max','owners_avg']].isnull().sum().sum()}")
display(df_limpo[["Estimated owners", "owners_min", "owners_max", "owners_avg"]].head(5))
print("✓ Três colunas numéricas criadas a partir da faixa em texto.")

EXTRAÇÃO NUMÉRICA DE ESTIMATED OWNERS
──────────────────────────────────────────────────
owners_min/max/avg ausentes: 0


,Estimated owners,owners_min,owners_max,owners_avg
0,0 - 0,0,0,0.0
1,0 - 20000,0,20000,10000.0
2,0 - 20000,0,20000,10000.0
3,0 - 20000,0,20000,10000.0
4,0 - 20000,0,20000,10000.0


✓ Três colunas numéricas criadas a partir da faixa em texto.


In [18]:
# TRATAMENTO 3 — Metacritic score: 0 (sentinela) -> NaN
zerados_antes = int((df_limpo["Metacritic score"] == 0).sum())
df_limpo["Metacritic score"] = df_limpo["Metacritic score"].replace(0, pd.NA)
zerados_depois = int((df_limpo["Metacritic score"] == 0).sum())

print("TRATAMENTO DO SENTINELA EM METACRITIC SCORE")
print("─" * 50)
print(f"Valores == 0 antes:  {zerados_antes}")
print(f"Valores == 0 depois: {zerados_depois}")
print(f"Valores ausentes (NaN) depois: {df_limpo['Metacritic score'].isna().sum()}")
print("✓ Sentinela convertido em ausência explícita; médias agora refletem só notas reais.")

TRATAMENTO DO SENTINELA EM METACRITIC SCORE
──────────────────────────────────────────────────
Valores == 0 antes:  121597
Valores == 0 depois: 0
Valores ausentes (NaN) depois: 121597
✓ Sentinela convertido em ausência explícita; médias agora refletem só notas reais.


In [19]:
# TRATAMENTO 4 — preencher ausências em colunas categóricas
colunas_categoricas = ["Genres", "Developers", "Categories", "Publishers", "Tags"]
ausentes_antes = df_limpo[colunas_categoricas].isnull().sum()

df_limpo[colunas_categoricas] = df_limpo[colunas_categoricas].fillna("Not specified")

print("PREENCHIMENTO DE CATEGÓRICAS AUSENTES")
print("─" * 50)
for coluna in colunas_categoricas:
    print(f"{coluna:12s}: {ausentes_antes[coluna]:6d} ausentes antes -> {df_limpo[coluna].isnull().sum()} depois")
print("✓ Marcador explícito 'Not specified' aplicado; nenhuma linha perdida em agrupamentos.")

PREENCHIMENTO DE CATEGÓRICAS AUSENTES
──────────────────────────────────────────────────
Genres      :   8423 ausentes antes -> 0 depois
Developers  :   8449 ausentes antes -> 0 depois
Categories  :   8966 ausentes antes -> 0 depois
Publishers  :   8922 ausentes antes -> 0 depois
Tags        :  42502 ausentes antes -> 0 depois
✓ Marcador explícito 'Not specified' aplicado; nenhuma linha perdida em agrupamentos.


In [20]:
# TRATAMENTO 5 — datas: converter Release date de texto para datetime real
tipo_antes = str(df_limpo["Release date"].dtype)
df_limpo["Release date"] = pd.to_datetime(df_limpo["Release date"], errors="coerce", format="mixed")
nao_convertidas = df_limpo["Release date"].isna().sum()

print("CONVERSÃO DE RELEASE DATE PARA DATETIME")
print("─" * 50)
print(f"Tipo antes:  {tipo_antes}")
print(f"Tipo depois: {df_limpo['Release date'].dtype}")
print(f"Datas que não puderam ser convertidas: {nao_convertidas}")
print("✓ Coluna de data pronta para análises temporais (ano de lançamento, sazonalidade etc.).")

CONVERSÃO DE RELEASE DATE PARA DATETIME
──────────────────────────────────────────────────
Tipo antes:  str
Tipo depois: datetime64[us]
Datas que não puderam ser convertidas: 0
✓ Coluna de data pronta para análises temporais (ano de lançamento, sazonalidade etc.).


In [21]:
# TRATAMENTO 6 — duplicatas de linha exata
shape_antes = df_limpo.shape
duplicatas_antes = int(df_limpo.duplicated().sum())
df_limpo = df_limpo.drop_duplicates().reset_index(drop=True)

print("TRATAMENTO DE DUPLICATAS EXATAS")
print("─" * 50)
print(f"Shape antes:            {shape_antes}")
print(f"Duplicatas encontradas: {duplicatas_antes}")
print(f"Shape depois:           {df_limpo.shape}")
if duplicatas_antes == 0:
    print("✓ Verificado: não havia duplicatas de linha exata nesta base (nada para remover).")
else:
    print(f"✓ {duplicatas_antes} linha(s) duplicada(s) removida(s).")

TRATAMENTO DE DUPLICATAS EXATAS
──────────────────────────────────────────────────
Shape antes:            (125855, 42)
Duplicatas encontradas: 0
Shape depois:           (125855, 42)
✓ Verificado: não havia duplicatas de linha exata nesta base (nada para remover).


# 7. Validar

Confirmamos que cada problema documentado foi de fato resolvido.

In [22]:
validacoes = pd.DataFrame({
    "regra verificada": [
        "Colunas alinhadas (About the game não é mais booleano/texto vazio genérico)",
        "Coluna Movies não existe mais",
        "owners_min/max/avg são numéricas",
        "Metacritic score não tem mais 0 como valor (só NaN ou nota real)",
        "Genres/Developers/Categories/Publishers/Tags sem ausências",
        "Release date é datetime",
        "Não há duplicatas de linha exata",
    ],
    "resultado": [
        pd.api.types.is_object_dtype(df_limpo["About the game"]) or pd.api.types.is_string_dtype(df_limpo["About the game"]),
        "Movies" not in df_limpo.columns,
        all(pd.api.types.is_numeric_dtype(df_limpo[c]) for c in ["owners_min", "owners_max", "owners_avg"]),
        (df_limpo["Metacritic score"] == 0).sum() == 0,
        df_limpo[["Genres", "Developers", "Categories", "Publishers", "Tags"]].isnull().sum().sum() == 0,
        pd.api.types.is_datetime64_any_dtype(df_limpo["Release date"]),
        df_limpo.duplicated().sum() == 0,
    ],
})
validacoes["status"] = validacoes["resultado"].map({True: "OK", False: "REVISAR"})

print("VALIDAÇÃO FINAL")
print("─" * 50)
display(validacoes[["regra verificada", "status"]].style.hide(axis="index"))

falhas = validacoes.loc[~validacoes["resultado"], "regra verificada"].tolist()
assert not falhas, f"Validações que falharam: {falhas}"
print("✓ Todas as regras definidas foram atendidas.")

VALIDAÇÃO FINAL
──────────────────────────────────────────────────


regra verificada,status
Colunas alinhadas (About the game não é mais booleano/texto vazio genérico),OK
Coluna Movies não existe mais,OK
owners_min/max/avg são numéricas,OK
Metacritic score não tem mais 0 como valor (só NaN ou nota real),OK
Genres/Developers/Categories/Publishers/Tags sem ausências,OK
Release date é datetime,OK
Não há duplicatas de linha exata,OK


✓ Todas as regras definidas foram atendidas.


In [23]:
print("RESUMO — ANTES (leitura ingênua) x DEPOIS (base preparada)")
print("─" * 60)
print(f"Shape antes:  {df_bruto.shape[0]} linhas × {df_bruto.shape[1]} colunas")
print(f"Shape depois: {df_limpo.shape[0]} linhas × {df_limpo.shape[1]} colunas")
print()
df_limpo.info()

RESUMO — ANTES (leitura ingênua) x DEPOIS (base preparada)
────────────────────────────────────────────────────────────
Shape antes:  125855 linhas × 39 colunas
Shape depois: 125855 linhas × 42 colunas



<class 'pandas.DataFrame'>
RangeIndex: 125855 entries, 0 to 125854
Data columns (total 42 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   AppID                       125855 non-null  int64         
 1   Name                        125854 non-null  str           
 2   Release date                125855 non-null  datetime64[us]
 3   Estimated owners            125855 non-null  str           
 4   Peak CCU                    125855 non-null  int64         
 5   Required age                125855 non-null  int64         
 6   Price                       125855 non-null  float64       
 7   DiscountDLC count           125855 non-null  int64         
 8   Discount                    125855 non-null  int64         
 9   About the game              117393 non-null  str           
 10  Supported languages         125855 non-null  str           
 11  Full audio languages        125855 non-null  str  

In [24]:
print("Valores ausentes por coluna, base preparada (só colunas com ausência):")
ausencias_final = df_limpo.isnull().sum()
display(ausencias_final[ausencias_final > 0].sort_values(ascending=False).to_frame("ausentes"))

Valores ausentes por coluna, base preparada (só colunas com ausência):


,ausentes
Score rank,125815
Metacritic score,121597
Metacritic url,121597
Reviews,113674
Notes,102545
Website,75335
Support url,70560
Support email,22630
About the game,8462
Screenshots,6027


In [25]:
df_limpo.to_csv(CAMINHO_LIMPO, index=False)
print(f"✓ Base preparada salva em: {CAMINHO_LIMPO}")
print(f"  {df_limpo.shape[0]} linhas × {df_limpo.shape[1]} colunas")

✓ Base preparada salva em: ../Dataset/fronkongames/games_clean.csv
  125855 linhas × 42 colunas


## Conclusão

A base `fronkongames/games.csv` chegou com um defeito estrutural sério (cabeçalho
malformado desalinhando quase todas as colunas), somado a problemas de
qualidade mais comuns (coluna vazia, tipo de dado errado, sentinela numérico
disfarçado de nota real, ausências em categóricas). Nenhum problema foi
inventado — todos foram confirmados com evidência antes do tratamento, e a
base preparada (`games_clean.csv`) está pronta para as próximas etapas do
projeto (análise exploratória e dashboard).